# Telecom Customer Churn — Preprocessing Notebook

This notebook produces the three CSV files needed by the main AutoML project:

- `output/train.csv` — training data (with the `Churn` target column)
- `output/sample_test.csv` — a small test sample **without** the target column
- `output/sample_test_labeled.csv` — the same sample **with** the target column

**Run order:** run every cell from top to bottom (menu `Run > Run All Cells`).

**Business problem.** A telecom company wants to predict which subscribers are about to cancel their plan (churn). Keeping a customer is much cheaper than acquiring a new one. The model output is `1` (will churn) or `0` (will stay).

## Step 1 — Imports and folder paths

Inside the Docker container, the working directory is `/home/jovyan/work`. The folders `data/` and `output/` are mounted from your machine, so anything written there appears on your host.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Folders (relative to /home/jovyan/work inside the container)
RAW_DIR = os.path.join('..', 'data', 'raw')
OUT_DIR = os.path.join('..', 'output')
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

RAW_CSV = os.path.join(RAW_DIR, 'telco_churn.csv')
print('Raw file target :', os.path.abspath(RAW_CSV))
print('Output folder   :', os.path.abspath(OUT_DIR))

## Step 2 — Download the dataset with `requests`

We download the public IBM Telco Customer Churn CSV. If the download fails (no internet in the container), the next cell explains how to place the file manually.

In [ ]:
import requests

URL = (
    'https://raw.githubusercontent.com/IBM/'
    'telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
)

if os.path.exists(RAW_CSV):
    print('Raw file already present — skipping download.')
else:
    try:
        print('Downloading from:', URL)
        resp = requests.get(URL, timeout=60)
        resp.raise_for_status()
        with open(RAW_CSV, 'wb') as f:
            f.write(resp.content)
        print('Download OK. Size:', len(resp.content), 'bytes')
    except Exception as exc:
        print('Download FAILED:', exc)
        print('See Step 2b below to place the file manually.')

### Step 2b — If the download failed (manual fallback)

If the cell above printed `Download FAILED`, download the file on your own machine instead and copy it into the `data/raw/` folder of this project, renamed to `telco_churn.csv`.

**Option A — PowerShell on your machine (host):**
```powershell
Invoke-WebRequest `
  -Uri "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv" `
  -OutFile "data\raw\telco_churn.csv"
```

**Option B — Kaggle:** download `WA_Fn-UseC_-Telco-Customer-Churn.csv` from the [Telco Customer Churn dataset](https://www.kaggle.com/datasets/blastchar/telco-customer-churn), then copy it into `data/raw/` and rename it to `telco_churn.csv`.

After placing the file, re-run all cells from the top.

## Step 3 — Load the raw data and look at it

In [ ]:
df = pd.read_csv(RAW_CSV)
print('Shape:', df.shape)
print('Columns:', list(df.columns))
df.head()

In [ ]:
# Quick look at types and the target distribution
print(df.dtypes)
print()
print('Churn value counts:')
print(df['Churn'].value_counts())

## Step 4 — Clean the data

Three problems to fix in this dataset:
1. `customerID` is just an identifier — not a predictor. Drop it.
2. `TotalCharges` is stored as text and contains blank spaces for brand-new customers. Convert to numeric.
3. `Churn` is `Yes`/`No` — convert to `1`/`0`.

In [ ]:
# 1) Drop the identifier column
df = df.drop(columns=['customerID'])

# 2) Fix TotalCharges: blanks -> NaN -> numeric -> fill missing with 0
#    (new customers with tenure 0 have an empty TotalCharges)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
n_missing = df['TotalCharges'].isna().sum()
print('Missing TotalCharges after conversion:', n_missing)
df['TotalCharges'] = df['TotalCharges'].fillna(0.0)

# 3) Encode the target Churn: Yes -> 1, No -> 0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0}).astype(int)

print('Churn distribution after encoding:')
print(df['Churn'].value_counts(normalize=True).round(3))

## Step 5 — One-hot encode the categorical columns

Every text column (gender, Contract, PaymentMethod, etc.) is turned into binary 0/1 columns. The numeric columns (`tenure`, `MonthlyCharges`, `TotalCharges`, `SeniorCitizen`) are left untouched.

In [ ]:
# Detect categorical columns automatically (object dtype), excluding the target
categorical_cols = [c for c in df.columns if df[c].dtype == 'object']
print('Categorical columns to encode:', categorical_cols)

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=False, dtype=int)

print('Shape before encoding:', df.shape)
print('Shape after encoding :', df_encoded.shape)
df_encoded.head()

## Step 6 — Split into train and test sets

We keep 90% for training and 10% for testing. `stratify=Churn` keeps the same churn ratio in both sets.

In [ ]:
TARGET = 'Churn'

train_df, test_df = train_test_split(
    df_encoded,
    test_size=0.10,
    random_state=42,
    stratify=df_encoded[TARGET],
)

# Keep the test sample small enough to demo nicely in the Streamlit UI
SAMPLE_SIZE = 50
sample_labeled = test_df.sample(n=min(SAMPLE_SIZE, len(test_df)), random_state=42)
sample_nolabel = sample_labeled.drop(columns=[TARGET])

print('train_df       :', train_df.shape)
print('sample_labeled :', sample_labeled.shape)
print('sample_nolabel :', sample_nolabel.shape)

## Step 7 — Save the three output files

These are written to the `output/` folder, which is mounted to your machine.

In [ ]:
train_path    = os.path.join(OUT_DIR, 'train.csv')
test_path     = os.path.join(OUT_DIR, 'sample_test.csv')
labeled_path  = os.path.join(OUT_DIR, 'sample_test_labeled.csv')

train_df.to_csv(train_path, index=False)
sample_nolabel.to_csv(test_path, index=False)
sample_labeled.to_csv(labeled_path, index=False)

print('Saved:')
print(' -', os.path.abspath(train_path))
print(' -', os.path.abspath(test_path))
print(' -', os.path.abspath(labeled_path))

## Step 8 — Final sanity checks

Confirm the files exist, that `train.csv` contains the `Churn` column, that `sample_test.csv` does **not**, and that `sample_test_labeled.csv` does.

In [ ]:
for p in [train_path, test_path, labeled_path]:
    exists = os.path.exists(p)
    cols = pd.read_csv(p, nrows=1).columns.tolist() if exists else []
    print(f'{os.path.basename(p):28s} exists={exists}  has_Churn={"Churn" in cols}  n_cols={len(cols)}')

print()
print('train.csv must have Churn      -> True expected')
print('sample_test.csv must NOT       -> False expected')
print('sample_test_labeled.csv must   -> True expected')

## Done!

Now copy the three files from `preprocessing-telecom-churn/output/` into the **main AutoML project**:

| From | To |
|---|---|
| `output/train.csv` | `backend/data/processed/train.csv` |
| `output/sample_test.csv` | `backend/data/sample_test.csv` |
| `output/sample_test_labeled.csv` | `backend/data/sample_test_labeled.csv` |

Then, in the main project, edit `docker-compose.yml` (`--target Churn`, `MODEL_NAME: telecom-churn-automl`) and `frontend/app.py` (`TARGET_COL`, `LABELS`), and run `docker compose up --build`.